# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.**

The question I framed in Week 1 — "which visible pages are underperforming their position
tier's typical CTR?" — sounds exactly like the first row of the framing skill's task-type
table: *"which ones first?"* → ranking/scoring, with a priority score as the target and
precision@K as the typical metric. It is not classification, because I am not predicting a
discrete future event from an observed outcome (like "will this page decline"); I am scoring
every eligible page on a continuous axis of "how much opportunity is likely sitting here right
now," so an editor can work down a sorted list. It is not clustering either — I already know
the axis I care about (CTR relative to expectation), so I don't need an unsupervised grouping
to discover it.

This matters for what I build next: a ranking/scoring task means the deliverable is an ordered
list with scores, not a binary yes/no flag, and it means the metric has to be rank-aware
(precision@K, or a correlation between my score and a later real outcome) rather than accuracy.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/UnsoundMouse/flyrankaiw01_research_question/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients")

Loaded: 30,000 rows x 44 columns | 32 clients


## 2. Target or proxy

**Target (proxy, not a fully observed outcome): `ctr_gap` = tier_median_ctr − ctr.**

A positive `ctr_gap` means a page's own CTR sits below what other pages at the same
`position_tier` typically achieve — the bigger the gap, the more "opportunity" I am claiming
sits on that page.

I am calling this a **proxy**, not an observed target, and saying so plainly because the
framing skill is explicit that "the target must be observed, not defined" — a label built from
my own rule risks teaching a future model to reproduce my rule rather than the world. The
honest version of an observed target for this lane would be something like *"did this page's
clicks or CTR rise in the 30 days after an editor rewrote its title/meta"* — a real, measured,
after-the-fact outcome. I don't have that yet: the starter CSV is a single 90-day snapshot with
no edit-event log, so there is no "before/after an edit" outcome to observe here.

Until that kind of outcome exists (or I build a proper past→future split on the warehouse
release in a later week), `ctr_gap` is a **stand-in I can compute today from currently
observed CTR and position**, useful for building and testing the ranking logic, but it is not
proof that a page flagged with a large gap will actually respond to editorial work. I keep this
distinction in the "careful words" section below.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the proxy target: ctr_gap = tier_median_ctr - ctr
# avg_position == 0 means "no data," not rank zero -- excluded before any tier comparison
eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)]

# tier medians computed on reasonably visible pages so low-volume noise doesn't dominate
visible = eligible[eligible["impressions_90d"] >= 500]
tier_median = visible.groupby("position_tier")["ctr"].median()

merged = eligible.merge(tier_median.rename("tier_median_ctr"), left_on="position_tier", right_index=True)
merged["ctr_gap"] = merged["tier_median_ctr"] - merged["ctr"]

print("ctr_gap summary (positive = underperforming its tier):")
print(merged["ctr_gap"].describe())
print()
print(f"Pages with a positive gap (candidate opportunities): "
      f"{(merged['ctr_gap'] > 0).sum():,} of {len(merged):,}")

ctr_gap summary (positive = underperforming its tier):
count    28795.000000
mean        -0.347687
std          3.228646
min        -99.910000
25%         -0.120000
50%          0.060000
75%          0.170000
max          0.240000
Name: ctr_gap, dtype: float64

Pages with a positive gap (candidate opportunities): 16,636 of 28,795


## 3. Success metric

**Primary metric (once a real outcome exists): precision@K** — of the top K pages my score
ranks highest, how many turn out to actually be worth an editor's time (measured by a later,
observed outcome such as post-edit click lift). This is the metric named in the framing skill
for ranking/scoring tasks, and it matches how the output is actually used: an editor only ever
works from the top of the list, so getting the top 20–50 right matters far more than
getting every page in the dataset correctly ordered.

**Interim metric (usable today, no outcome label needed yet): rank correlation** between
`ctr_gap` and an independent signal that plausibly reflects real momentum, as a sanity check
that the proxy isn't just noise. I used `trend_pct` for this check *only as an outside sanity
signal*, never as an input feature — the data dictionary flags `trend_pct` and
`trend_direction` as label-trap columns computed downstream of the same information I'd be
trying to predict, so they can never be used inside the scoring logic itself, only as an
external check on whether the proxy behaves sensibly.

I will report both types honestly: the interim correlation check now, and precision@K once a
real, observed validation outcome is defined (Week 3–4 territory, using the warehouse release's
time structure).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from scipy.stats import spearmanr

# Sanity check ONLY -- trend_pct is never a feature (label-trap column per the data dictionary),
# used here purely as an external signal to see if ctr_gap behaves sensibly.
sub = merged.dropna(subset=["ctr_gap", "trend_pct"])
corr, pval = spearmanr(sub["ctr_gap"], sub["trend_pct"])

print(f"Spearman correlation, ctr_gap vs trend_pct (external sanity check, n={len(sub):,}):")
print(f"  rho = {corr:.3f}, p = {pval:.4g}")
print()
print("Interpretation: a real (though modest) negative relationship -- pages with a bigger")
print("current CTR gap tend to skew toward weaker trend_pct. This does not validate the proxy")
print("as causal or even necessarily useful, but it shows ctr_gap isn't pure noise: it lines up")
print("with an independent signal in a sensible direction.")

Spearman correlation, ctr_gap vs trend_pct (external sanity check, n=26,604):
  rho = -0.212, p = 3.275e-269

Interpretation: a real (though modest) negative relationship -- pages with a bigger
current CTR gap tend to skew toward weaker trend_pct. This does not validate the proxy
as causal or even necessarily useful, but it shows ctr_gap isn't pure noise: it lines up
with an independent signal in a sensible direction.


## 4. The unit of analysis, as a real dataframe

**One row = one content item for one client, at the 90-day snapshot in the starter CSV** — a
`(client_id, content_id)` pair. This is the grain the whole lane operates at: I am not scoring
queries, sessions, or days — I am scoring *pages*, because the action ("rewrite this page's
title/meta") is a page-level action.

The grain probe below confirms `(client_id, content_id)` is actually unique in this dataset —
an assumption I need to check rather than assume, per the data skill's warning that IDs are
pseudonyms meant for grouping/joining, and that grain should always be probed, not trusted.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# [A] GRAIN PROBE — is (client_id, content_id) really unique?
grain_counts = df.groupby(["client_id", "content_id"]).size()
print(f"Max rows per (client_id, content_id): {grain_counts.max()}")
print(f"Any duplicate grain? {(grain_counts > 1).any()}")
print(f"Unique (client_id, content_id) pairs: {len(grain_counts):,} vs total rows: {len(df):,}")
print()

# [B] The unit of analysis as an actual dataframe
cols = ["client_id", "content_id", "content_type", "main_intent",
        "avg_position", "position_tier", "impressions_90d", "ctr",
        "tier_median_ctr", "ctr_gap"]
print("One row = one (client_id, content_id) page, with its proxy target attached:")
merged[cols].sort_values("ctr_gap", ascending=False).head(5)

Max rows per (client_id, content_id): 1
Any duplicate grain? False
Unique (client_id, content_id) pairs: 30,000 vs total rows: 30,000

One row = one (client_id, content_id) page, with its proxy target attached:


,client_id,content_id,content_type,main_intent,avg_position,position_tier,impressions_90d,ctr,tier_median_ctr,ctr_gap
18894,client_d029fa3a95,content_776fb785a163,keyword article,informational,6.7,page_1,38,0.0,0.24,0.24
29973,client_d4735e3a26,content_4be930227848,feedly article,NaN,3.5,page_1,2,0.0,0.24,0.24
18905,client_8722616204,content_3a00be9a1f6b,keyword article,transactional,9.6,page_1,239,0.0,0.24,0.24
6,client_8722616204,content_9a34b442b552,keyword article,informational,7.0,page_1,20,0.0,0.24,0.24
29977,client_d029fa3a95,content_c87291853cab,comparison article,informational,8.2,page_1,112,0.0,0.24,0.24


## 5. Why ML beats a fixed rule here

It doesn't yet — and that's the honest, useful answer at this stage. The tier-median rule
built in Week 1 *is* a fixed rule, and it already produces a defensible, position-adjusted
candidate list. Right now I have no evidence a learned model would beat it, because I haven't
trained one or validated either approach against a real outcome.

Where a fixed single-signal rule (position tier alone) genuinely runs out of room, based on
what's visible in this data:

- **The tier bucket is coarse.** Five tiers collapse a continuous `avg_position` into five
  buckets, and pages near a tier boundary (position 5 vs. position 6, `top_3` vs `striking`)
  get compared against very different medians despite being nearly identical in reality.
- **Other signals plausibly matter but aren't in the rule at all.** `content_type`,
  `main_intent`, `word_count_tier`, and `freshness_tier` all vary across the flagged pages (see
  the sample above — `keyword article` pages with different intents show up with very different
  gaps), and a rule using position alone has no way to account for that.
- **The interim correlation check** shows `ctr_gap` already lines up with an outside signal in
  a sensible direction — meaning there's a real pattern here worth modeling, not noise a more
  complex method would just be overfitting to.

What ML would earn its place doing, if the evidence supports it: learning a smoother, several-
signal expectation of "typical CTR for a page like this" instead of five hard-coded tier
buckets, and outputting a confidence-ranked score instead of a single hard cutoff. But the
honest claim right now is: *the tier-median rule is the baseline to beat, not a strawman* — if a
trained model can't clearly outperform it on precision@K against a real outcome, the rule wins
and that is a legitimate result, not a failed project.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Names the ML task type (ranking / scoring), the target/proxy, and the success metric
- [x] Shows the unit of analysis as a real dataframe (one row = one client's content page)
- [x] Explains why this is an ML/analysis problem and not just a rule — honestly, without
      overclaiming ML wins yet
- [x] Ties the output to a real content action (editor rewrites title/meta on top-ranked pages)
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.